# DengueWatch LK
### Explainable Machine Learning for Next-Week Dengue Case Forecasting in Sri Lanka

**Student No: 269161X**

This project applies **Extra Trees Regressor**, a machine learning algorithm not covered in the module lectures, to forecast next-week dengue case counts for Sri Lankan reporting areas. The model is evaluated using a chronological train/validation/test design and explained using SHAP and LIME.

In [ ]:
!pip install -q shap lime joblib

## 1. Problem Definition and Dataset

The problem is to forecast the number of dengue cases expected in the **next reporting week** for each Sri Lankan reporting area.

- **Data source:** Sri Lankan weekly dengue case data compiled from publicly available surveillance data.
- **Main fields:** year, week, start date, end date, district/reporting area and cases.
- **Target:** `target_next_week`.
- **Ethics:** the dataset contains aggregated weekly case counts and no personal identifiers or sensitive individual-level records.

The forecasting task is useful because an early indication of an increase or decrease in cases can support planning and awareness. This is an academic prototype and not an official public-health warning system.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('srilanka_weekly_data.csv')
df['date'] = pd.to_datetime(df['start.date'], format='%m/%d/%Y')
df = df.sort_values(['district','date']).reset_index(drop=True)

print('Rows and columns:', df.shape)
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
print('Reporting areas:', df['district'].nunique())
print('Missing values in main columns:')
print(df[['year','week','start.date','district','cases']].isna().sum())
print('Duplicate district-date rows:', df.duplicated(['district','date']).sum())
df.head()

In [ ]:
print(df['cases'].describe())

annual = df.groupby(df['date'].dt.year)['cases'].sum()
plt.figure(figsize=(10,4))
plt.plot(annual.index, annual.values, marker='o')
plt.title('Total Reported Dengue Cases by Year')
plt.xlabel('Year')
plt.ylabel('Cases')
plt.grid(alpha=0.2)
plt.show()

## 2. Preprocessing and Feature Engineering

The raw weekly series is converted into a supervised regression dataset. Lag values and rolling averages summarize recent dengue activity, while sine/cosine transformations represent annual seasonality.

Only rows where the following record is approximately one week later (6–9 days) are retained, so the target represents a true next-week observation.

In [ ]:
grp = df.groupby('district')

for lag in [1,2,3,4,8,12,52]:
    df[f'cases_lag_{lag}'] = grp['cases'].shift(lag)

for w in [4,8,12]:
    df[f'rolling_mean_{w}'] = grp['cases'].transform(
        lambda s: s.shift(1).rolling(w).mean()
    )

df['trend_current_4'] = df['cases'] - df['cases_lag_4']
df['week_sin'] = np.sin(2*np.pi*df['week']/52.0)
df['week_cos'] = np.cos(2*np.pi*df['week']/52.0)

df['target_next_week'] = grp['cases'].shift(-1)
df['next_date'] = grp['date'].shift(-1)
df['gap_to_target'] = (df['next_date'] - df['date']).dt.days

ml = df[(df['gap_to_target'] >= 6) & (df['gap_to_target'] <= 9)].dropna().copy()

features = [
    'district','week','week_sin','week_cos','cases',
    'cases_lag_1','cases_lag_2','cases_lag_3','cases_lag_4',
    'cases_lag_8','cases_lag_12','cases_lag_52',
    'rolling_mean_4','rolling_mean_8','rolling_mean_12',
    'trend_current_4'
]

print('ML-ready rows:', len(ml))
ml[features + ['target_next_week']].head()

## 3. Selection of the New Machine Learning Algorithm

### Extra Trees Regressor (Extremely Randomized Trees)

Extra Trees was selected because the dengue forecasting problem contains nonlinear relationships and interactions among recent case counts, lagged values, rolling averages, seasonality and reporting area.

**Why it is suitable**
- It can learn nonlinear relationships without assuming a fixed linear form.
- It handles many interacting numerical features well.
- It averages many randomized trees, which can reduce variance compared with a single tree.
- It works directly for regression and is compatible with post-hoc explainability methods such as SHAP and LIME.

**How it differs from standard models**
- **Decision Tree:** one tree chooses splits greedily; Extra Trees averages many trees and introduces stronger randomness.
- **Random Forest:** both are tree ensembles, but Extra Trees uses more randomized split thresholds rather than searching as extensively for the best split at each node.
- **Logistic Regression:** logistic regression is mainly for classification and models a parametric relationship; Extra Trees is non-parametric and can capture nonlinear interactions.
- **k-NN:** k-NN predicts from nearby observations at inference time, while Extra Trees learns an ensemble of randomized decision rules during training.

The lecture-material audit for this assignment found no evidence that Extra Trees was covered in the module lectures.

## 4. Chronological Train, Validation and Test Split

To avoid future information leaking into the past, the data is split by time rather than randomly:

- **Training:** up to 2022
- **Validation:** 2023–2024
- **Final test:** 2025

The validation period is used for hyperparameter selection. After selecting the configuration, the final model is retrained on training + validation data and evaluated once on the untouched 2025 test set.

In [ ]:
train = ml[ml['date'].dt.year <= 2022].copy()
val = ml[ml['date'].dt.year.isin([2023, 2024])].copy()
test = ml[ml['date'].dt.year == 2025].copy()

print('Train:', len(train))
print('Validation:', len(val))
print('Test:', len(test))

## 5. Preprocessing and Hyperparameter Selection

`district` is one-hot encoded. The remaining engineered features are already numeric.

A small set of Extra Trees configurations is compared **only on the validation set**. RMSE is used as the main selection metric, while MAE and R² are also reported.

Hyperparameters considered:
- `n_estimators`: number of randomized trees.
- `max_depth`: limits tree complexity.
- `min_samples_leaf`: prevents very small terminal leaves.
- `max_features`: controls the proportion of features considered at each split.
- `random_state=42`: provides reproducibility.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import ExtraTreesRegressor

pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['district']),
    ('num', 'passthrough', [c for c in features if c != 'district'])
])

configs = {
    'ET-A': dict(n_estimators=80, max_depth=16, min_samples_leaf=3, max_features=0.8),
    'ET-B': dict(n_estimators=120, max_depth=18, min_samples_leaf=3, max_features=0.8),
    'ET-C': dict(n_estimators=150, max_depth=18, min_samples_leaf=3, max_features=0.8),
    'ET-D': dict(n_estimators=180, max_depth=20, min_samples_leaf=2, max_features=0.8)
}

X_train_trans = pre.fit_transform(train[features])
X_val_trans = pre.transform(val[features])

rows = []
models = {}

for name, params in configs.items():
    model = ExtraTreesRegressor(**params, n_jobs=-1, random_state=42)
    model.fit(X_train_trans, train['target_next_week'])
    pred = np.maximum(0, model.predict(X_val_trans))
    rows.append([
        name,
        mean_absolute_error(val['target_next_week'], pred),
        mean_squared_error(val['target_next_week'], pred) ** 0.5,
        r2_score(val['target_next_week'], pred)
    ])
    models[name] = model

validation_results = pd.DataFrame(rows, columns=['Configuration','MAE','RMSE','R2'])
display(validation_results.round(4).sort_values('RMSE'))

### Hyperparameter Choice

The selected configuration is **ET-C**. It gives strong validation performance while keeping the saved model compact enough for GitHub and Streamlit deployment:

- `n_estimators = 150`
- `max_depth = 18`
- `min_samples_leaf = 3`
- `max_features = 0.8`

In the validation period, ET-C achieved approximately **MAE 13.27, RMSE 30.19 and R² 0.8359**. Increasing model complexity produced only very small validation changes while creating a much larger model file, so ET-C was selected as the practical accuracy/deployment trade-off.

## 6. Final Model Training and Evaluation

After hyperparameter selection, the model is retrained using **training + validation data**. The 2025 test set remains untouched until this final evaluation.

A simple **last-week persistence benchmark** is also shown. This is not a second machine-learning algorithm; it simply predicts that next week's cases will equal the current week's cases.

In [ ]:
train_val = pd.concat([train, val], ignore_index=True)

final_pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['district']),
    ('num', 'passthrough', [c for c in features if c != 'district'])
])

final_model = Pipeline([
    ('preprocessor', final_pre),
    ('model', ExtraTreesRegressor(
        n_estimators=150,
        max_depth=18,
        min_samples_leaf=3,
        max_features=0.8,
        n_jobs=-1,
        random_state=42
    ))
])

final_model.fit(train_val[features], train_val['target_next_week'])

pred_test = np.maximum(0, final_model.predict(test[features]))
y_test = test['target_next_week']

final_results = pd.DataFrame([
    ['Extra Trees Regressor',
     mean_absolute_error(y_test, pred_test),
     mean_squared_error(y_test, pred_test) ** 0.5,
     r2_score(y_test, pred_test)],
    ['Last-week benchmark',
     mean_absolute_error(y_test, test['cases']),
     mean_squared_error(y_test, test['cases']) ** 0.5,
     r2_score(y_test, test['cases'])]
], columns=['Method','MAE','RMSE','R2'])

display(final_results.round(4))

### Final Test Interpretation

After retraining on training + validation data, Extra Trees achieved approximately:

- **MAE: 8.35 cases**
- **RMSE: 16.13 cases**
- **R²: 0.9273**

The model improves MAE over the last-week benchmark, while the benchmark remains slightly better on RMSE/R². This indicates that recent dengue case counts are already a very strong predictor of the following week. The result is reported transparently rather than claiming that the ML model dominates every metric.

In [ ]:
plt.figure(figsize=(10,4))
plt.scatter(y_test, pred_test, alpha=0.35)
limit = max(y_test.max(), pred_test.max())
plt.plot([0, limit], [0, limit], linestyle='--')
plt.xlabel('Actual next-week cases')
plt.ylabel('Predicted next-week cases')
plt.title('Extra Trees: Actual vs Predicted on 2025 Test Set')
plt.show()

## 7. Explainability with SHAP

SHAP is used for both global and local explanations. The global summary identifies the features that most influence forecasts overall. The waterfall plot explains one individual prediction.

In [ ]:
import shap

pre_fitted = final_model.named_steps['preprocessor']
model_fitted = final_model.named_steps['model']

X_sample = test[features].sample(min(300, len(test)), random_state=42)
X_trans = pre_fitted.transform(X_sample)

feature_names = pre_fitted.get_feature_names_out()
clean_feature_names = [
    name.replace('num__', '').replace('cat__', '')
    for name in feature_names
]

explainer = shap.TreeExplainer(model_fitted)
shap_values = explainer(X_trans)

shap.summary_plot(
    shap_values.values,
    X_trans,
    feature_names=clean_feature_names
)

In [ ]:
shap_values_named = shap.Explanation(
    values=shap_values.values,
    base_values=shap_values.base_values,
    data=X_trans,
    feature_names=clean_feature_names
)

shap.plots.waterfall(shap_values_named[0], max_display=10)

### SHAP Interpretation

The expected pattern is that current/recent dengue case counts, short-term lag values and rolling averages have the strongest influence, while seasonal week features provide additional context.

This behavior is reasonable for a one-week-ahead surveillance forecast because dengue counts usually have strong short-term persistence. However, SHAP values describe model attribution, **not causal effects**.

## 8. Local Explanation with LIME

LIME explains a selected prediction by fitting a simple local surrogate around that observation.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

X_train_val_trans = pre_fitted.transform(train_val[features])
X_test_trans = pre_fitted.transform(test[features])

lime_explainer = LimeTabularExplainer(
    X_train_val_trans,
    feature_names=clean_feature_names,
    mode='regression',
    random_state=42
)

lime_result = lime_explainer.explain_instance(
    X_test_trans[0],
    model_fitted.predict,
    num_features=10
)

lime_df = pd.DataFrame(
    lime_result.as_list(),
    columns=['Feature', 'Contribution']
)

display(lime_df)

plt.figure(figsize=(8,5))
plt.barh(lime_df['Feature'], lime_df['Contribution'])
plt.xlabel('Contribution to prediction')
plt.ylabel('Feature')
plt.title('LIME Explanation for One Prediction')
plt.gca().invert_yaxis()
plt.show()

### LIME Interpretation

The LIME explanation shows which feature conditions increase or decrease one selected forecast. It complements SHAP by providing a local, instance-level explanation.

## 9. Critical Discussion

### Limitations
- The model uses historical dengue counts and calendar information only; it does not include rainfall, temperature, humidity, mosquito indices, mobility or intervention data.
- 2026 data is partial and is not used for final performance evaluation.
- Forecast performance may change during unusual outbreaks or structural changes in reporting.

### Data Quality
- Surveillance data can be affected by reporting delays, under-reporting, changes in case definitions and administrative differences between reporting areas.
- `Kalmune` appears as a separate reporting label, so the dataset contains 26 reporting labels rather than exactly 25 standard districts.

### Bias and Fairness
- The model may perform differently across reporting areas because case volumes and reporting practices differ.
- Lower-volume districts may have fewer outbreak examples, which can make predictions less stable.
- The system should not be used to allocate healthcare resources automatically without expert review.

### Real-World Impact and Ethics
- A useful forecast could support planning, awareness and prioritization.
- Incorrect forecasts could create false reassurance or unnecessary concern.
- Predictions should therefore be treated as decision support, not as official public-health advice.
- The dataset is aggregated and contains no personal identifiers, which reduces privacy risk.

### Alignment with Domain Knowledge
The strong influence of recent case counts and short-term rolling trends is consistent with the temporal persistence expected in weekly disease surveillance. Seasonal features are also plausible because dengue transmission varies across time. These associations should not be interpreted as proof of causation.

## 10. Save Model and Results

The trained pipeline is saved so the same preprocessing and Extra Trees model can be used by the Streamlit application.

In [ ]:
import joblib

joblib.dump(final_model, 'best_model_extratrees.joblib')
validation_results.to_csv('extratrees_validation_results.csv', index=False)
final_results.to_csv('extratrees_final_test_results.csv', index=False)

print('Saved: best_model_extratrees.joblib')
print('Saved: extratrees_validation_results.csv')
print('Saved: extratrees_final_test_results.csv')